# Prepare daily explanatory variables for regression models

To build new time series models (regressions, accumulated precipitation) or to study model errors,
we need to process additional ERA5-Land products:
- soil water "sw",
- precipitations "tp", 
- soil runoff "sro"
- soil reservoir content "src".

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

In [ ]:
from etdataset.api import (
    parse_date,
    prepare_daily_explanatory,
)
from etdataset.era5 import (
    create_daily_explanatory_dataset,
    download_date_by_date,
    read,
)
from etdataset.interpolation import create_grid_dataset
from etdataset.utils import get_utm_bbox_from_roi
from etdataset.writer import write_daily_explanatory


### Input

In [ ]:
# Acquisition dates
min_date_str = "2024-03-01"  # Format YYYY-MM-DD
max_date_str = "2024-03-31"  # Format YYYY-MM-DD


In [ ]:
min_date = parse_date(min_date_str)
max_date = parse_date(max_date_str)

In [ ]:
roi_path = os.path.join("data", "Zone_Senegal_Centre.shp")

In [ ]:
output_path = os.path.join("out")
os.makedirs(output_path, exist_ok=True)

In [ ]:
roi_bbox, roi_crs = get_utm_bbox_from_roi(roi_path)
grid = create_grid_dataset(roi_bbox, roi_crs, 3000)

### Downloading Era5-Land for the date range

In [ ]:
date_list_prec = download_date_by_date(
    min_date,
    max_date,
    [
        "total_precipitation",
        "surface_runoff",
        "skin_reservoir_content",
        "volumetric_soil_water_layer_1",
    ],
    output_path,
)

### Selecting a date to explore and examine the daily evolution of variables

In [ ]:
date = date_list_prec[0]

In [ ]:
exp_brute = read("out/ERA5_data/download_era5land_2024-01-01.zip")

In [ ]:
exp_brute

#### Surface runoff

In [ ]:
exp_brute.sro.sel(latitude=43.55, longitude=1.09, method="nearest").plot()

#### Precipitations

In [ ]:
exp_brute.tp.sel(latitude=43.55, longitude=1.09, method="nearest").plot()

#### Soil reservoir content

In [ ]:
exp_brute.src.sel(latitude=48.23, longitude=4.29, method="nearest").plot()

#### Volumtric soil water

In [ ]:
exp_brute.swvl1.sel(latitude=48.23, longitude=4.29, method="nearest").plot()

### Reading the content of the .zip directory as dataset

In [ ]:
dst_prec = create_daily_explanatory_dataset(date, grid, output_path)

In [ ]:
dst_prec

### Plotting the spatial distribution of the variables over the ROI

In [ ]:
dst_prec.tp.plot()

In [ ]:
dst_prec.sro.plot()

In [ ]:
dst_prec.src.plot()

In [ ]:
dst_prec.sw.plot()

### Creating a "explanatory_YYMMDD.tif" file from the dataset

In [ ]:
write_daily_explanatory(dst_prec, output_path)

### Preparing explanatory files for the entire date range

In [ ]:
prepare_daily_explanatory(min_date_str, max_date_str, roi_path, output_path)